In [15]:
import pandas as pd
import re

# 1. Load the local workbook
file_path = 'Debits May 23 (1).xlsx'
xls = pd.ExcelFile(file_path)

In [16]:
# 2. Load the sheets
# It will look for 'Data' and 'Vendor List' specifically
df_data = pd.read_excel(xls, sheet_name='Data')
df_mapping = pd.read_excel(xls, sheet_name='Vendor List')

# Clean up any accidental whitespace in your mapping sheet headers/text
df_mapping.columns = [str(col).strip() for col in df_mapping.columns]
df_mapping = df_mapping.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [17]:
# 3. The Cleaning & Mapping Function
def get_clean_vendor(description):
    if not isinstance(description, str):
        return "Unknown"
        
    # Check against your local Vendor List sheet first
    for _, row in df_mapping.iterrows():
        match_text = str(row.get('If Includes', '')).strip()
        if match_text and match_text.lower() in description.lower():
            return row['Vendor Name']
            
    # Fallback Cleaning: Remove dates and normalize giant spaces
    cleaned = description
    cleaned = re.sub(r'\b\d{2}/\d{2}\b', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

In [18]:
# 4. Create the 'Vendor' column
df_data['Vendor Name'] = df_data['Description'].apply(get_clean_vendor)

# 5. Bring in Category and Sub-Category from your local list
# We drop 'Match' to keep the merge clean
mapping_subset = df_mapping[['Vendor Name', 'Category', 'Sub-Category']].drop_duplicates(subset=['Vendor Name'])
df_final = pd.merge(df_data, mapping_subset, on='Vendor Name', how='left')

In [19]:
# 6. Fill in defaults for unmapped items
df_final['Category'] = df_final['Category'].fillna('Uncategorized')
df_final['Sub-Category'] = df_final['Sub-Category'].fillna('Uncategorized')

In [20]:
# 7. Save the final file locally
output_file = 'Categorized_Debits_Master_May_23 (1).xlsx'
df_final.to_excel(output_file, index=False)

print(f"Done! Created '{output_file}' with {len(df_final)} rows processed.")

Done! Created 'Categorized_Debits_Master_May_23 (1).xlsx' with 1504 rows processed.
